GSE274880: HS and normal skin

source: https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE274880

In [ ]:
!pip install aeacus

In [ ]:
import os
import urllib.request
import tarfile
import scanpy as sc
import anndata as ad
import aeacus
import numpy as np
import pandas as pd

In [ ]:
DATA_DIR = '/home/pandeyps/Prefix/aeacus/data/GSE274880'
os.makedirs(DATA_DIR, exist_ok=True)
os.chdir(DATA_DIR)
print(f"Working directory: {os.getcwd()}")

In [ ]:
RAW_URL = 'https://ftp.ncbi.nlm.nih.gov/geo/series/GSE274nnn/GSE274880/suppl/GSE274880_RAW.tar'
RAW_TAR = os.path.join(DATA_DIR, 'GSE274880_RAW.tar')

In [ ]:
!curl -L -o {RAW_TAR} {RAW_URL}

In [ ]:
with tarfile.open(RAW_TAR, 'r') as tar:
    tar.extractall(path=DATA_DIR)

In [ ]:
h5_files = sorted([f for f in os.listdir(DATA_DIR) if f.endswith('.h5')])

adatas = []
sample_map = {
    'GSM8459837': 'Normal_Skin_1',
    'GSM8459838': 'Normal_Skin_2',
    'GSM8459839': 'HS_Skin_1',
    'GSM8459840': 'HS_Skin_2',
    'GSM8459841': 'HS_Skin_3',
    'GSM8459842': 'HS_Skin_4',
    'GSM8459843': 'HS_Skin_5',
    'GSM8459844': 'HS_Skin_6',
}

In [ ]:
for h5_file in h5_files:
    h5_path = os.path.join(DATA_DIR, h5_file)
    gsm_id = h5_file.replace('.h5', '')
    sample_name = sample_map.get(gsm_id, gsm_id)
    print(f"Loading {h5_file} as {sample_name}...")
    adata = sc.read_10x_h5(h5_path)
    adata.obs_names = [f"{gsm_id}_{b}_{i}" for i, b in enumerate(adata.obs_names)]
    adata.obs['sample'] = sample_name
    if not adata.var_names.is_unique:
        adata.var_names_make_unique()
    adata.obs['condition'] = 'HS' if 'HS' in sample_name else 'Normal'
    adata.obs['gsm_id'] = gsm_id
    adatas.append(adata)
    print(f"  Cells: {adata.n_obs}, Genes: {adata.n_vars}")

adata = ad.concat(adatas, merge='same')

In [ ]:
sc.pp.calculate_qc_metrics(adata, percent_top=None, log1p=False, inplace=True)
sc.pp.filter_cells(adata, min_genes=200)
sc.pp.filter_genes(adata, min_cells=3)

In [23]:
TEMP_H5AD = os.path.join(DATA_DIR, 'GSE274880_combined.h5ad')
adata.write_h5ad(TEMP_H5AD)

result = aeacus.Profiler(test_input=TEMP_H5AD, norm_type="cpm_log1p").load().profile()

print(f"Malignant cells: {(result.obs['malignancy_call'] == 'Malignant').sum()}")
print(f"Non-malignant cells: {(result.obs['malignancy_call'] == 'Normal').sum()}")

Model features: 3778
Missing features: 43 (1.14%)
Malignant cells: 132
Non-malignant cells: 24534
